# 04 Build Modelling Datasets

## Purpose
Build the main modelling-ready datasets required for the later recommendation and prediction stages.

This notebook combines the outputs of the earlier interaction-cleaning and recipe-preprocessing phases.

## Objectives
- load the cleaned interaction datasets
- load the cleaned joined recipe dataset
- confirm the expected columns and shapes
- build the explicit modelling dataset
- build the implicit modelling dataset
- build the joined interaction–recipe modelling table
- check join coverage and missing recipe-side features
- save the modelling datasets for later phases

## Prior findings carried into this phase

`02_interaction_cleaning.ipynb` established that:
- cleaned interaction data were standardised and date-parsed
- `rating = 0` should be treated as an observed but unrated interaction
- separate explicit and implicit interaction datasets were created

`03_recipe_preprocessing.ipynb` established that:
- `RAW_recipes.csv` should remain the base recipe catalogue
- `PP_recipes.csv` should only be used as an enrichment layer
- the cleaned recipe tables were joined on `recipe_id`
- the joined recipe table preserves the full raw recipe catalogue

This notebook therefore builds separate modelling datasets for recommendation experiments while preserving the earlier preprocessing decisions.

In [19]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

In [20]:
PROJECT_ROOT = Path.cwd().resolve().parents[0]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [21]:
from src.paths import (
    PROCESSED_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    ensure_directories,
)

ensure_directories()

print("Output directories are ready.")

Output directories are ready.


## 1. Loading processed datasets

Load the cleaned interaction outputs and the joined recipe table.

The following processed inputs are used:
- `interactions_clean.parquet`
- `interactions_explicit.parquet`
- `interactions_implicit.parquet`
- `recipes_joined.parquet`

In [22]:
interactions_clean = pd.read_parquet(PROCESSED_DIR / "interactions_clean.parquet")
interactions_explicit = pd.read_parquet(PROCESSED_DIR / "interactions_explicit.parquet")
interactions_implicit = pd.read_parquet(PROCESSED_DIR / "interactions_implicit.parquet")
recipes_joined = pd.read_parquet(PROCESSED_DIR / "recipes_joined.parquet")

print("Clean interactions shape:", interactions_clean.shape)
print("Explicit interactions shape:", interactions_explicit.shape)
print("Implicit interactions shape:", interactions_implicit.shape)
print("Joined recipes shape:", recipes_joined.shape)

Clean interactions shape: (1132367, 9)
Explicit interactions shape: (1071520, 9)
Implicit interactions shape: (1132367, 8)
Joined recipes shape: (231637, 37)


In [23]:
display(interactions_clean.head(3))
display(interactions_explicit.head(3))
display(interactions_implicit.head(3))
display(recipes_joined.head(3))

,user_id,recipe_id,date,rating,review,review_exists,is_unrated_observation,explicit_rating,implicit_feedback
0,2008,992,2000-01-25,5,better than any you can get at a restaurant!,1,0,5.0,1
1,2008,3603,2000-01-25,4,better than having to actually GO to a Hooters! ;-),1,0,4.0,1
2,2046,517,2000-02-25,5,thought this was terrific!,1,0,5.0,1


,user_id,recipe_id,date,rating,review,review_exists,is_unrated_observation,explicit_rating,implicit_feedback
0,2008,992,2000-01-25,5,better than any you can get at a restaurant!,1,0,5,1
1,2008,3603,2000-01-25,4,better than having to actually GO to a Hooters! ;-),1,0,4,1
2,2046,517,2000-02-25,5,thought this was terrific!,1,0,5,1


,user_id,recipe_id,date,rating,review,review_exists,is_unrated_observation,implicit_feedback
0,2008,992,2000-01-25,5,better than any you can get at a restaurant!,1,0,1
1,2008,3603,2000-01-25,4,better than having to actually GO to a Hooters! ;-),1,0,1
2,2046,517,2000-02-25,5,thought this was terrific!,1,0,1


,recipe_id,name,minutes,contributor_id,submitted,description,n_steps,n_ingredients,name_length,description_length,tag_count,step_count_from_list,ingredient_count_from_list,nutrition_vector_length,nutrition_sum,nutrition_mean,has_description,has_tags,has_steps,has_ingredients,tag_breakfast,tag_lunch,tag_dinner,tag_dessert,tag_healthy,tag_easy,tag_vegetarian,tag_vegan,i,calorie_level,name_token_count,ingredient_token_group_count,step_token_count,technique_vector_length,technique_count_active,ingredient_id_count,has_pp_features
0,137739,arriba baked winter squash mexican style,55,47892,2005-09-16,"autumn is my favorite time of year to cook! this recipe \r\ncan be prepared either spicy or sweet, your choice!\r\ntwo of my posted mexican-inspired seasoning mix recipes are offered as suggestions.",11,7,42,194,20,11,7,7,70.5,10.071429,1,1,1,1,0,0,0,0,0,1,1,0,145702,0,9,7,165,58,3,7,1.0
1,31490,a bit different breakfast pizza,30,26278,2002-06-17,this recipe calls for the crust to be prebaked a bit before adding ingredients. feel free to change sausage to ham or bacon. this warms well in the microwave for those late risers.,9,6,32,180,20,9,6,7,266.4,38.057143,1,1,1,1,1,0,0,0,0,1,0,0,33090,0,7,6,101,58,3,6,1.0
2,112140,all in the kitchen chili,130,196586,2005-02-25,this modified version of 'mom's' chili was a hit at our 2004 christmas party. we made an extra large pot to have some left to freeze but it never made it to the freezer. it was a favorite by all. ...,6,13,25,295,9,6,13,7,442.8,63.257143,1,1,1,1,0,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN


## 2. Confirming the available columns

Before creating the modelling datasets, confirm that the expected columns are present in each processed input.

In [24]:
print("Clean interaction columns:")
print(interactions_clean.columns.tolist())

print("\nExplicit interaction columns:")
print(interactions_explicit.columns.tolist())

print("\nImplicit interaction columns:")
print(interactions_implicit.columns.tolist())

print("\nJoined recipe columns:")
print(recipes_joined.columns.tolist())

Clean interaction columns:
['user_id', 'recipe_id', 'date', 'rating', 'review', 'review_exists', 'is_unrated_observation', 'explicit_rating', 'implicit_feedback']

Explicit interaction columns:
['user_id', 'recipe_id', 'date', 'rating', 'review', 'review_exists', 'is_unrated_observation', 'explicit_rating', 'implicit_feedback']

Implicit interaction columns:
['user_id', 'recipe_id', 'date', 'rating', 'review', 'review_exists', 'is_unrated_observation', 'implicit_feedback']

Joined recipe columns:
['recipe_id', 'name', 'minutes', 'contributor_id', 'submitted', 'description', 'n_steps', 'n_ingredients', 'name_length', 'description_length', 'tag_count', 'step_count_from_list', 'ingredient_count_from_list', 'nutrition_vector_length', 'nutrition_sum', 'nutrition_mean', 'has_description', 'has_tags', 'has_steps', 'has_ingredients', 'tag_breakfast', 'tag_lunch', 'tag_dinner', 'tag_dessert', 'tag_healthy', 'tag_easy', 'tag_vegetarian', 'tag_vegan', 'i', 'calorie_level', 'name_token_count', 'in

## 3. Building the explicit modelling dataset

The explicit dataset is intended for rating-aware experiments such as explicit matrix factorisation or rating analysis.

It should retain only rows where an explicit rating exists and keep the columns needed for explicit recommendation modelling.

In [25]:
explicit_columns = [
    "user_id",
    "recipe_id",
    "date",
    "rating",
    "explicit_rating",
    "review_exists",
    "implicit_feedback",
]

explicit_model_df = interactions_explicit[explicit_columns].copy()

explicit_model_df["user_id"] = explicit_model_df["user_id"].astype("int64")
explicit_model_df["recipe_id"] = explicit_model_df["recipe_id"].astype("int64")
explicit_model_df["explicit_rating"] = explicit_model_df["explicit_rating"].astype("int64")
explicit_model_df["implicit_feedback"] = explicit_model_df["implicit_feedback"].astype("int8")
explicit_model_df["review_exists"] = explicit_model_df["review_exists"].astype("int8")

print("Explicit modelling dataset shape:", explicit_model_df.shape)
explicit_model_df.head()

Explicit modelling dataset shape: (1071520, 7)


,user_id,recipe_id,date,rating,explicit_rating,review_exists,implicit_feedback
0,2008,992,2000-01-25,5,5,1,1
1,2008,3603,2000-01-25,4,4,1,1
2,2046,517,2000-02-25,5,5,1,1
3,2046,4523,2000-02-25,2,2,1,1
4,2046,4684,2000-02-25,5,5,1,1


## 4. Validating the explicit modelling dataset

Check that the explicit dataset only contains valid explicit ratings and no unrated observations.

In [26]:
print("Null explicit_rating rows:", int(explicit_model_df["explicit_rating"].isna().sum()))
print("Unique explicit ratings:", sorted(explicit_model_df["explicit_rating"].unique().tolist()))
print("Date range:", explicit_model_df["date"].min(), "to", explicit_model_df["date"].max())
print("Unique users:", explicit_model_df["user_id"].nunique())
print("Unique recipes:", explicit_model_df["recipe_id"].nunique())

Null explicit_rating rows: 0
Unique explicit ratings: [1, 2, 3, 4, 5]
Date range: 2000-01-25 00:00:00 to 2018-12-20 00:00:00
Unique users: 196098
Unique recipes: 226590


## 5. Building the implicit modelling dataset

The implicit dataset is intended for Top-N recommendation experiments.

It retains all observed interactions, including rows where `rating = 0`, because those rows were previously defined as observed but unrated interactions rather than explicit negative feedback.

In [27]:
implicit_columns = [
    "user_id",
    "recipe_id",
    "date",
    "rating",
    "review_exists",
    "is_unrated_observation",
    "implicit_feedback",
]

implicit_model_df = interactions_implicit[implicit_columns].copy()

implicit_model_df["user_id"] = implicit_model_df["user_id"].astype("int64")
implicit_model_df["recipe_id"] = implicit_model_df["recipe_id"].astype("int64")
implicit_model_df["rating"] = implicit_model_df["rating"].astype("int64")
implicit_model_df["review_exists"] = implicit_model_df["review_exists"].astype("int8")
implicit_model_df["is_unrated_observation"] = implicit_model_df["is_unrated_observation"].astype("int8")
implicit_model_df["implicit_feedback"] = implicit_model_df["implicit_feedback"].astype("int8")

print("Implicit modelling dataset shape:", implicit_model_df.shape)
implicit_model_df.head()

Implicit modelling dataset shape: (1132367, 7)


,user_id,recipe_id,date,rating,review_exists,is_unrated_observation,implicit_feedback
0,2008,992,2000-01-25,5,1,0,1
1,2008,3603,2000-01-25,4,1,0,1
2,2046,517,2000-02-25,5,1,0,1
3,2046,4523,2000-02-25,2,1,0,1
4,2046,4684,2000-02-25,5,1,0,1


## 6. Validating the implicit modelling dataset

Check that all implicit rows have observed interaction feedback and inspect the treatment of unrated observations.

In [28]:
print("Implicit feedback values:", sorted(implicit_model_df["implicit_feedback"].unique().tolist()))
print("Unrated observation count:", int(implicit_model_df["is_unrated_observation"].sum()))
print(
    "Unrated observation percentage:",
    round(implicit_model_df["is_unrated_observation"].mean() * 100, 2),
)
print("Date range:", implicit_model_df["date"].min(), "to", implicit_model_df["date"].max())
print("Unique users:", implicit_model_df["user_id"].nunique())
print("Unique recipes:", implicit_model_df["recipe_id"].nunique())

Implicit feedback values: [1]
Unrated observation count: 60847
Unrated observation percentage: 5.37
Date range: 2000-01-25 00:00:00 to 2018-12-20 00:00:00
Unique users: 226570
Unique recipes: 231637


## 7. Building the joined interaction–recipe modelling table

The joined table is intended for later feature engineering, Random Forest dataset construction, dashboard display support, and hybrid extensions.

The join is performed on `recipe_id` and uses the cleaned recipe table as the recipe-side source.

In [29]:
interaction_recipe_joined = interactions_clean.merge(
    recipes_joined,
    on="recipe_id",
    how="left",
    validate="many_to_one",
)

print("Joined interaction–recipe table shape:", interaction_recipe_joined.shape)
interaction_recipe_joined.head(5)

Joined interaction–recipe table shape: (1132367, 45)


,user_id,recipe_id,date,rating,review,review_exists,is_unrated_observation,explicit_rating,implicit_feedback,name,minutes,contributor_id,submitted,description,n_steps,n_ingredients,name_length,description_length,tag_count,step_count_from_list,ingredient_count_from_list,nutrition_vector_length,nutrition_sum,nutrition_mean,has_description,has_tags,has_steps,has_ingredients,tag_breakfast,tag_lunch,tag_dinner,tag_dessert,tag_healthy,tag_easy,tag_vegetarian,tag_vegan,i,calorie_level,name_token_count,ingredient_token_group_count,step_token_count,technique_vector_length,technique_count_active,ingredient_id_count,has_pp_features
0,2008,992,2000-01-25,5,better than any you can get at a restaurant!,1,0,5.0,1,jalapeno pepper poppers,30,1545,1999-09-06,"originally from ""taste of home"" magazine",11,10,23,40,12,11,10,7,168.4,24.057143,1,1,1,1,0,0,0,0,0,0,0,0,80837,0,9,10,103,58,2,10,1.0
1,2008,3603,2000-01-25,4,better than having to actually GO to a Hooters! ;-),1,0,4.0,1,hooters buffalo wings,27,2353,1999-09-24,from top secret recipes http://www.topsecretrecipes.com,16,12,21,55,28,16,12,7,1257.1,179.585714,1,1,1,1,0,0,0,0,0,1,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN
2,2046,517,2000-02-25,5,thought this was terrific!,1,0,5.0,1,greek stuffed meatloaf,149,1609,1999-08-23,this is one of my favorite greek recipes.,15,20,22,41,20,15,20,7,555.4,79.342857,1,1,1,1,0,0,0,0,0,0,0,0,87844,1,5,20,202,58,3,20,1.0
3,2046,4523,2000-02-25,2,i think i did something wrong because i could taste the cornstarch in the finished product.,1,0,2.0,1,chinese imperial palace general tso s chicken,60,1932,1999-11-15,yum,16,15,45,3,19,16,15,7,702.7,100.385714,1,1,1,1,0,0,0,0,1,0,0,0,96521,1,10,15,187,58,5,15,1.0
4,2046,4684,2000-02-25,5,this is absolutely delicious. i even served it with lime slices so you could squeeze on more of the acid.,1,0,5.0,1,flank steak with lime chipotle sauce,75,1533,1999-11-18,yum,6,10,36,3,19,6,10,7,897.8,128.257143,1,1,1,1,0,0,0,0,0,1,0,0,44367,1,10,10,88,58,5,10,1.0


## 8. Checking join coverage

The earlier audit and recipe-preprocessing phases established that the recipe side should preserve the full raw catalogue.

This section confirms that the modelling join behaves as expected and measures whether any interaction rows fail to find a recipe match.

In [30]:
missing_recipe_rows = interaction_recipe_joined["name"].isna().sum()
missing_recipe_pct = round(missing_recipe_rows / len(interaction_recipe_joined) * 100, 4)

join_summary = pd.DataFrame([
    {
        "interaction_rows": int(len(interaction_recipe_joined)),
        "unique_interaction_users": int(interaction_recipe_joined["user_id"].nunique()),
        "unique_interaction_recipes": int(interaction_recipe_joined["recipe_id"].nunique()),
        "missing_recipe_rows": int(missing_recipe_rows),
        "missing_recipe_rows_pct": missing_recipe_pct,
    }
])

join_summary

,interaction_rows,unique_interaction_users,unique_interaction_recipes,missing_recipe_rows,missing_recipe_rows_pct
0,1132367,226570,231637,1,0.0001


In [31]:
print("Rows with missing recipe-side match:", int(missing_recipe_rows))
print("Percentage missing recipe-side match:", missing_recipe_pct)

Rows with missing recipe-side match: 1
Percentage missing recipe-side match: 0.0001


## 9. Inspecting recipe-side feature availability after the join

Because the recipe preprocessing phase used `RAW_recipes` as the base table and `PP_recipes` as an enrichment layer, some joined rows may still lack PP-derived fields.

This section measures the availability of those enrichment features in the joined table.

In [32]:
pp_feature_summary = pd.DataFrame([
    {
        "rows_with_pp_features": int(interaction_recipe_joined["has_pp_features"].fillna(0).sum()),
        "rows_without_pp_features": int((interaction_recipe_joined["has_pp_features"].fillna(0) == 0).sum()),
        "rows_with_pp_features_pct": round(
            interaction_recipe_joined["has_pp_features"].fillna(0).mean() * 100,
            2,
        ),
    }
])

pp_feature_summary

,rows_with_pp_features,rows_without_pp_features,rows_with_pp_features_pct
0,935120,197247,82.58


In [33]:
selected_join_columns = [
    "user_id",
    "recipe_id",
    "date",
    "rating",
    "explicit_rating",
    "implicit_feedback",
    "name",
    "minutes",
    "n_steps",
    "n_ingredients",
    "tag_count",
    "calorie_level",
    "num_techniques",
    "has_pp_features",
]

available_selected_join_columns = [
    col for col in selected_join_columns if col in interaction_recipe_joined.columns
]

display(interaction_recipe_joined[available_selected_join_columns].head(10))

,user_id,recipe_id,date,rating,explicit_rating,implicit_feedback,name,minutes,n_steps,n_ingredients,tag_count,calorie_level,has_pp_features
0,2008,992,2000-01-25,5,5.0,1,jalapeno pepper poppers,30,11,10,12,0,1.0
1,2008,3603,2000-01-25,4,4.0,1,hooters buffalo wings,27,16,12,28,<NA>,NaN
2,2046,517,2000-02-25,5,5.0,1,greek stuffed meatloaf,149,15,20,20,1,1.0
3,2046,4523,2000-02-25,2,2.0,1,chinese imperial palace general tso s chicken,60,16,15,19,1,1.0
4,2046,4684,2000-02-25,5,5.0,1,flank steak with lime chipotle sauce,75,6,10,19,1,1.0
5,1773,278,2000-03-13,4,4.0,1,greek spinach triangles,125,18,8,18,0,1.0
6,1773,7435,2000-03-13,5,5.0,1,kevin s best corned beef,275,19,16,19,2,1.0
7,2046,3431,2000-04-07,5,5.0,1,leek tomato goat cheese pizza,0,12,6,15,2,1.0
8,2046,13307,2000-05-21,5,5.0,1,neiman marcus 250 chocolate chip cookies recipe,36,7,13,18,0,1.0
9,2156,148,2000-06-02,0,NaN,1,bugwiches,40,8,11,26,<NA>,NaN


## 10. Preparing compact model-specific views from the joined table

The joined interaction–recipe table is useful as a broad master table, but later phases may benefit from smaller task-specific views.

This section creates a compact joined table that keeps the key interaction fields and the most useful recipe-side descriptive features.

In [34]:
joined_model_columns = [
    "user_id",
    "recipe_id",
    "date",
    "rating",
    "explicit_rating",
    "review_exists",
    "is_unrated_observation",
    "implicit_feedback",
    "name",
    "minutes",
    "submitted",
    "n_steps",
    "n_ingredients",
    "name_length",
    "description_length",
    "tag_count",
    "step_count_from_list",
    "ingredient_count_from_list",
    "nutrition_vector_length",
    "nutrition_sum",
    "nutrition_mean",
    "has_description",
    "calorie_level",
    "num_name_tokens",
    "num_steps_tokens",
    "num_ingredient_ids",
    "num_techniques",
    "has_pp_features",
]

joined_model_columns = [
    col for col in joined_model_columns if col in interaction_recipe_joined.columns
]

joined_model_df = interaction_recipe_joined[joined_model_columns].copy()

print("Compact joined modelling dataset shape:", joined_model_df.shape)
joined_model_df.head()

Compact joined modelling dataset shape: (1132367, 24)


,user_id,recipe_id,date,rating,explicit_rating,review_exists,is_unrated_observation,implicit_feedback,name,minutes,submitted,n_steps,n_ingredients,name_length,description_length,tag_count,step_count_from_list,ingredient_count_from_list,nutrition_vector_length,nutrition_sum,nutrition_mean,has_description,calorie_level,has_pp_features
0,2008,992,2000-01-25,5,5.0,1,0,1,jalapeno pepper poppers,30,1999-09-06,11,10,23,40,12,11,10,7,168.4,24.057143,1,0,1.0
1,2008,3603,2000-01-25,4,4.0,1,0,1,hooters buffalo wings,27,1999-09-24,16,12,21,55,28,16,12,7,1257.1,179.585714,1,<NA>,NaN
2,2046,517,2000-02-25,5,5.0,1,0,1,greek stuffed meatloaf,149,1999-08-23,15,20,22,41,20,15,20,7,555.4,79.342857,1,1,1.0
3,2046,4523,2000-02-25,2,2.0,1,0,1,chinese imperial palace general tso s chicken,60,1999-11-15,16,15,45,3,19,16,15,7,702.7,100.385714,1,1,1.0
4,2046,4684,2000-02-25,5,5.0,1,0,1,flank steak with lime chipotle sauce,75,1999-11-18,6,10,36,3,19,6,10,7,897.8,128.257143,1,1,1.0


## 11. Summarising the modelling datasets

Create a simple summary table covering the three main outputs from this phase.

In [35]:
modelling_summary = pd.DataFrame([
    {
        "dataset": "explicit_model_df",
        "rows": int(len(explicit_model_df)),
        "users": int(explicit_model_df["user_id"].nunique()),
        "recipes": int(explicit_model_df["recipe_id"].nunique()),
        "min_date": explicit_model_df["date"].min(),
        "max_date": explicit_model_df["date"].max(),
    },
    {
        "dataset": "implicit_model_df",
        "rows": int(len(implicit_model_df)),
        "users": int(implicit_model_df["user_id"].nunique()),
        "recipes": int(implicit_model_df["recipe_id"].nunique()),
        "min_date": implicit_model_df["date"].min(),
        "max_date": implicit_model_df["date"].max(),
    },
    {
        "dataset": "joined_model_df",
        "rows": int(len(joined_model_df)),
        "users": int(joined_model_df["user_id"].nunique()),
        "recipes": int(joined_model_df["recipe_id"].nunique()),
        "min_date": joined_model_df["date"].min(),
        "max_date": joined_model_df["date"].max(),
    },
])

modelling_summary

,dataset,rows,users,recipes,min_date,max_date
0,explicit_model_df,1071520,196098,226590,2000-01-25,2018-12-20
1,implicit_model_df,1132367,226570,231637,2000-01-25,2018-12-20
2,joined_model_df,1132367,226570,231637,2000-01-25,2018-12-20


## 12. Saving modelling outputs

The explicit, implicit, and joined modelling tables are saved for later phases.

These outputs will be used by:
- chronological splitting
- collaborative filtering and matrix factorisation experiments
- later feature engineering
- dashboard integration

In [36]:
explicit_model_df.to_parquet(PROCESSED_DIR / "model_explicit.parquet", index=False)
implicit_model_df.to_parquet(PROCESSED_DIR / "model_implicit.parquet", index=False)
joined_model_df.to_parquet(PROCESSED_DIR / "model_interaction_recipe_joined.parquet", index=False)

modelling_summary.to_csv(TABLES_DIR / "04_modelling_dataset_summary.csv", index=False)
join_summary.to_csv(TABLES_DIR / "04_join_coverage_summary.csv", index=False)
pp_feature_summary.to_csv(TABLES_DIR / "04_pp_feature_availability_summary.csv", index=False)

print("Saved modelling datasets.")

Saved modelling datasets.


## 13. Final decisions

The following modelling-dataset decisions were applied:

- the explicit modelling dataset retains only rows with valid explicit ratings
- the implicit modelling dataset retains all observed interactions, including unrated observations where `rating = 0`
- the joined interaction–recipe table uses `recipe_id` as the merge key
- the recipe-side join preserves the earlier recipe-preprocessing decision to use the joined recipe table as the master metadata source
- compact saved outputs are created for later model training and feature engineering

